In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
# Importing dataset
dataset = pd.read_csv("Social_Network_Ads.csv")
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [3]:
# Handling categorical data
dataset = pd.get_dummies(dataset,drop_first=True,dtype=int)
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [4]:
# Dropping the 'USER ID' Columns as it is unique identifier
dataset = dataset.drop("User ID",axis=1)
dataset

,Age,EstimatedSalary,Purchased,Gender_Male
0,19,19000,0,1
1,35,20000,0,1
2,26,43000,0,0
3,27,57000,0,0
4,19,76000,0,1
...,...,...,...,...
395,46,41000,1,0
396,51,23000,1,1
397,50,20000,1,0
398,36,33000,0,1


In [5]:
dataset.columns

Index(['Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [6]:
dataset['Purchased'].value_counts()

Purchased
0    257
1    143
Name: count, dtype: int64

In [7]:
# Feature and target selection
independent = dataset[['Age', 'EstimatedSalary', 'Gender_Male']]
dependent = dataset['Purchased']

In [8]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [9]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [10]:
# Splitting data into training and test sets
x_train,x_test,y_train,y_test = train_test_split(independent,dependent,test_size=0.3,random_state=0)

In [11]:
x_train.shape

(280, 3)

In [12]:
x_test.shape

(120, 3)

In [13]:
y_train.shape

(280,)

In [14]:
y_test.shape

(120,)

In [15]:
# Data Standardization
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [16]:
# GridSearch 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# Defining the different parameters for gridsearch optimization
param_grid = {'n_neighbors':[3,5,7],
              'weights':['uniform','distance'],
              'algorithm':['auto','kd_tree','ball_tree','brute']}

grid = GridSearchCV(KNeighborsClassifier(),param_grid,refit=True,verbose=3,scoring='f1_weighted',n_jobs=-1)
grid.fit(x_train,y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'algorithm': ['auto', 'kd_tree', 'ball_tree', 'brute'],
                         'n_neighbors': [3, 5, 7],
                         'weights': ['uniform', 'distance']},
             scoring='f1_weighted', verbose=3)

In [17]:
# Best gridsearch parameter
print(grid.best_params_)

{'algorithm': 'auto', 'n_neighbors': 7, 'weights': 'uniform'}


In [18]:
# Grid prediction
y_pred = grid.predict(x_test)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 1, 0, 0, 1])

In [19]:
# evaluation metrics
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[73  6]
 [ 4 37]]


In [20]:
from sklearn.metrics import classification_report
report = classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.92      0.94        79
           1       0.86      0.90      0.88        41

    accuracy                           0.92       120
   macro avg       0.90      0.91      0.91       120
weighted avg       0.92      0.92      0.92       120



In [21]:
from sklearn.metrics import f1_score
f1_weighted= f1_score(y_test,y_pred,average='weighted')
print(f1_weighted)

0.9171245421245421


In [22]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,grid.predict_proba(x_test)[:,1])

np.float64(0.9424205001543686)

In [23]:
results = grid.cv_results_
table = pd.DataFrame.from_dict(results)
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_algorithm,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.006343,0.002956,0.019623,0.001580,auto,3,uniform,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.892857,0.874254,0.841398,0.911105,0.910254,0.885974,0.026049,17
1,0.004730,0.001414,0.007836,0.001044,auto,3,distance,"{'algorithm': 'auto', 'n_neighbors': 3, 'weigh...",0.874254,0.874254,0.824293,0.893717,0.891667,0.871637,0.025075,21
2,0.003506,0.000659,0.015425,0.001306,auto,5,uniform,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.874254,0.874254,0.876643,0.929144,0.946153,0.900089,0.031147,5
3,0.002579,0.000189,0.007255,0.001194,auto,5,distance,"{'algorithm': 'auto', 'n_neighbors': 5, 'weigh...",0.855314,0.855314,0.859435,0.929144,0.946153,0.889072,0.040054,13
4,0.003771,0.001167,0.014916,0.001252,auto,7,uniform,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.874254,0.874254,0.859435,0.947015,0.964286,0.903849,0.042989,1
5,0.002996,0.000705,0.006878,0.000801,auto,7,distance,"{'algorithm': 'auto', 'n_neighbors': 7, 'weigh...",0.855314,0.874254,0.859435,0.929144,0.946153,0.892860,0.037496,9
6,0.002409,0.000133,0.012610,0.001399,kd_tree,3,uniform,"{'algorithm': 'kd_tree', 'n_neighbors': 3, 'we...",0.892857,0.874254,0.841398,0.911105,0.910254,0.885974,0.026049,17
7,0.002784,0.000311,0.007556,0.001640,kd_tree,3,distance,"{'algorithm': 'kd_tree', 'n_neighbors': 3, 'we...",0.874254,0.874254,0.824293,0.893717,0.891667,0.871637,0.025075,21
8,0.002594,0.000462,0.013433,0.000420,kd_tree,5,uniform,"{'algorithm': 'kd_tree', 'n_neighbors': 5, 'we...",0.874254,0.874254,0.876643,0.929144,0.946153,0.900089,0.031147,5
9,0.003234,0.001759,0.006871,0.000934,kd_tree,5,distance,"{'algorithm': 'kd_tree', 'n_neighbors': 5, 'we...",0.855314,0.855314,0.859435,0.929144,0.946153,0.889072,0.040054,13


In [24]:
age = int(input("Enter your age:"))
salary = float(input("Enter your salary"))
gender = int(input("male=1,female=0"))

In [25]:
data = sc.transform([[age,salary,gender]])

C:\Users\thiru\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [26]:
result = grid.predict(data)
result

array([0])